# 강의 03 · 실습 1 — 에이전트 동작 원리 · (6) 고난도 III

## 1. 문제상황

- 구름월드 고객센터에 「환불 규정이랑 주차 안내를 정리해서 알려 주세요」처럼 여러 항목을 한 번에 정리해 달라는 요청이 들어옵니다.
- 지금의 안내 프로그램은 질문을 받자마자 도구를 부르기 시작하므로, 요청이 몇 가지 일로 이루어졌는지 미리 정리하지 않습니다.
- 그래서 항목이 여러 개일 때 하나를 빠뜨리기도 하고, 무엇을 했고 무엇이 남았는지 실행 기록만 봐서는 알기 어렵습니다.
- 담당자는 프로그램이 먼저 할 일 목록을 세우고, 목록의 한 줄씩 처리한 뒤, 결과를 모아 한 단락으로 정리해 주기를 원합니다.

## 2. 문제와 목표

- **문제**: 여러 항목을 묻는 요청을 계획 없이 처리하면 항목을 빠뜨리기 쉽고, 실행 기록에서 진행 상황을 읽기 어렵습니다.
- **목표**
  - 요청을 받으면 모델이 먼저 할 일 목록(3단계 이내)을 세웁니다.
  - 목록의 한 줄마다 도구 호출 루프를 돌려 답을 얻고, 단계별 결과를 모아 모델이 한 단락으로 종합하는 프로그램을 만듭니다.
    - 세 부분: 계획 수립(도구 없는 모델 호출), 단계별 실행(줄마다 도구 호출 루프 한 바퀴), 종합(도구 없는 모델 호출)
  - 주어지는 것: FAQ 사전 세 항목과 현재 시각 도구(도구 호출 루프는 단계 ①~④ 구성 그대로). 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
  - 요청은 「환불 규정이랑 주차 안내를 정리해서 알려 주세요.」 하나이고, 반복 상한은 4회입니다.
  - 할 일 목록은 도구 없이 모델을 불러 「한 줄에 하나씩, 번호 없이, 3단계 이내」로 받고, 종합도 도구 없이 모델을 불러 한 단락으로 받습니다.
  - 계획 생성 호출의 최소 코드:

    ```python
    plan_res = llm.invoke(f"다음 요청을 처리할 단계 목록을 한 줄에 하나씩, 번호 없이 출력하라. 3단계 이내.\n요청: {question}")
    steps = [s.strip() for s in plan_res.content.splitlines() if s.strip()]
    # 줄마다 run_agent(step)을 돌려 결과를 모으고, 마지막에 llm.invoke로 한 단락으로 종합합니다.
    ```
- **목표 달성 여부의 판정 기준**:
  - 출력에 할 일 목록이 먼저 찍히고, 목록의 줄마다 도구 호출 기록이 찍히며,
  - 마지막 답 한 단락에 환불 규정과 주차 안내가 모두 들어 있는 것을 확인합니다.
  - 출력에는 번호를 붙인 할 일 목록, 줄마다의 도구 호출 기록과 단계 결과, 마지막 답 한 단락이 찍힙니다.

## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 주어진 것

라이브러리 불러오기, `.env` 읽기, 모델 준비와 주어진 자료는 아래 셀에 있습니다. 셀을 고치지 않고 그대로 실행합니다.

- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, `OPENAI_API_KEY=발급받은_키` 한 줄만 넣습니다.

In [ ]:
import os

from datetime import datetime
from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

# 주어진 자료
FAQ = {
    "운영시간": "매일 09:30~21:00에 운영합니다.",
    "주차": "주차장은 4,000대 규모이며 최초 30분은 무료입니다.",
    "환불": "이용일 전날까지 전액 환불, 당일은 50% 환불입니다.",
}


In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 출력의 맨 앞에 번호가 붙은 할 일 목록이 3줄 이내로 찍힙니다. 환불 규정 조회와 주차 안내 조회가 서로 다른 줄에 들어 있습니다.
2. 목록의 줄마다 그 줄을 처리한 도구 호출 기록과 단계 결과가 찍힙니다. 단계마다 도구 호출 루프가 따로 돌았다는 뜻입니다.
3. 마지막 답 한 단락에 환불 규정(전날까지 전액, 당일 50%)과 주차 안내(4,000대, 최초 30분 무료)가 모두 들어 있습니다.

세 가지가 모두 확인되면 완성입니다. 할 일 목록의 줄 수와 문장은 실행할 때마다 달라질 수 있습니다.